In [1]:
# ============================================================
# KIDNEY DISEASE / CKD STAGE PREDICTION
# RANDOM FOREST CLASSIFICATION
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import pickle
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 2. DATASET PATH
# ============================================================

TRAIN_PATH = r"A:\Dataset\Training_CKD_dataset.csv"
TEST_PATH = r"A:\Dataset\Testing_CKD_dataset.csv"

# If your files are in another location, change the above paths.


# ============================================================
# 3. LOAD DATASET
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape :", test_df.shape)


# ============================================================
# 4. DISPLAY DATA
# ============================================================

print("\nTraining Dataset:")
print(train_df.head())

print("\nTarget Classes:")
print(train_df["Target"].value_counts())


# ============================================================
# 5. SEPARATE FEATURES AND TARGET
# ============================================================

X_train = train_df.drop("Target", axis=1)
y_train = train_df["Target"]

X_test = test_df.drop("Target", axis=1)
y_test = test_df["Target"]


# ============================================================
# 6. ENCODE CATEGORICAL FEATURES
# ============================================================

categorical_columns = [
    "Diabetes",
    "Hypertension",
    "Smoking_Status",
    "Family_History_Kidney"
]

feature_encoders = {}

for column in categorical_columns:

    encoder = LabelEncoder()

    # Combine train + test values
    combined_values = pd.concat(
        [X_train[column], X_test[column]]
    ).astype(str)

    encoder.fit(combined_values)

    X_train[column] = encoder.transform(
        X_train[column].astype(str)
    )

    X_test[column] = encoder.transform(
        X_test[column].astype(str)
    )

    feature_encoders[column] = encoder


# ============================================================
# 7. ENCODE TARGET
# ============================================================

target_encoder = LabelEncoder()

y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)


# ============================================================
# 8. DISPLAY TARGET ENCODING
# ============================================================

print("\nTarget Encoding:")

for number, label in enumerate(target_encoder.classes_):
    print(number, "=", label)


# ============================================================
# 9. CREATE RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)


# ============================================================
# 10. TRAIN MODEL
# ============================================================

print("\nTraining Random Forest Model...")

model.fit(
    X_train,
    y_train_encoded
)

print("MODEL TRAINING COMPLETED!")


# ============================================================
# 11. PREDICTION
# ============================================================

y_pred = model.predict(X_test)


# ============================================================
# 12. ACCURACY
# ============================================================

accuracy = accuracy_score(
    y_test_encoded,
    y_pred
)

print("\n========================================")
print("MODEL PERFORMANCE")
print("========================================")

print("Accuracy:", accuracy)

print("Accuracy Percentage:",
      round(accuracy * 100, 2), "%")


# ============================================================
# 13. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test_encoded,
        y_pred,
        target_names=target_encoder.classes_
    )
)


# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

print("\nConfusion Matrix:")

cm = confusion_matrix(
    y_test_encoded,
    y_pred
)

print(cm)


# ============================================================
# 15. FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n========================================")
print("TOP IMPORTANT FEATURES")
print("========================================")

print(feature_importance.head(15))


# ============================================================
# 16. SAVE MODEL
# ============================================================

model_data = {
    "model": model,
    "feature_encoders": feature_encoders,
    "target_encoder": target_encoder,
    "feature_columns": X_train.columns.tolist()
}

MODEL_PATH = "kidney_disease_model.pkl"

with open(MODEL_PATH, "wb") as file:
    pickle.dump(model_data, file)


print("\n========================================")
print("MODEL SAVED SUCCESSFULLY")
print("========================================")

print("Saved File:", MODEL_PATH)

Training Dataset Shape: (21000, 36)
Testing Dataset Shape : (4800, 36)

Training Dataset:
                 Target  Age  Gender  BMI  Systolic_BP  Diastolic_BP  \
0        Healthy Kidney   29       1   28           97            69   
1  Severe CKD (Stage 4)   43       0   18          165           100   
2        Healthy Kidney   77       0   32          116            63   
3        Healthy Kidney   83       0   24           93            75   
4        Healthy Kidney   38       1   19          111            70   

   Heart_Rate  Serum_Creatinine  Blood_Urea_Nitrogen  eGFR  ...  \
0          99                 0                   12    95  ...   
1          67                 5                   87    28  ...   
2         101                 0                   16   100  ...   
3          87                 0                   10   101  ...   
4          92                 0                   10   102  ...   

   Fasting_Glucose     HbA1c  Cholesterol  Triglycerides  Serum_Albumin  \

In [1]:
# ============================================================
# KIDNEY DISEASE PREDICTION
# TRAIN + TEST + SAMPLE PREDICTION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# ============================================================
# 1. DATASET PATH
# ============================================================

TRAIN_PATH = r"A:\Dataset\Training_CKD_dataset.csv"
TEST_PATH = r"A:\Dataset\Testing_CKD_dataset.csv"


# ============================================================
# 2. LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training Shape:", train_df.shape)
print("Testing Shape :", test_df.shape)


# ============================================================
# 3. TARGET
# ============================================================

TARGET = "Target"

X_train = train_df.drop(TARGET, axis=1).copy()
y_train = train_df[TARGET].astype(str)

X_test = test_df.drop(TARGET, axis=1).copy()
y_test = test_df[TARGET].astype(str)


# ============================================================
# 4. ENCODE CATEGORICAL COLUMNS
# ============================================================

encoders = {}

for column in X_train.columns:

    # If column is text
    if X_train[column].dtype == "object":

        encoder = LabelEncoder()

        # Fit using training values
        encoder.fit(
            X_train[column].astype(str)
        )

        X_train[column] = encoder.transform(
            X_train[column].astype(str)
        )

        X_test[column] = encoder.transform(
            X_test[column].astype(str)
        )

        encoders[column] = encoder


# ============================================================
# 5. ENCODE TARGET
# ============================================================

target_encoder = LabelEncoder()

y_train_encoded = target_encoder.fit_transform(
    y_train
)

y_test_encoded = target_encoder.transform(
    y_test
)


# ============================================================
# 6. TRAIN RANDOM FOREST
# ============================================================

print("\n========================================")
print("TRAINING MODEL...")
print("========================================")

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(
    X_train,
    y_train_encoded
)

print("MODEL TRAINING COMPLETED!")


# ============================================================
# 7. TEST MODEL
# ============================================================

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test_encoded,
    predictions
)

print("\n========================================")
print("MODEL ACCURACY")
print("========================================")

print(
    f"Accuracy: {accuracy * 100:.2f}%"
)


# ============================================================
# 8. TAKE ONE REAL TEST PATIENT
# ============================================================

patient = X_test.iloc[[0]]

actual_value = y_test.iloc[0]


# ============================================================
# 9. PREDICT
# ============================================================

prediction = model.predict(
    patient
)

predicted_value = target_encoder.inverse_transform(
    prediction
)[0]


# ============================================================
# 10. CONFIDENCE
# ============================================================

probabilities = model.predict_proba(
    patient
)[0]

confidence = np.max(
    probabilities
) * 100


# ============================================================
# 11. FINAL OUTPUT
# ============================================================

print("\n\n")
print("========================================")
print("       KIDNEY DISEASE PREDICTION")
print("========================================")

print(
    "\nActual Disease:"
)

print(
    f"➡️ {actual_value}"
)

print(
    "\nPredicted Disease:"
)

print(
    f"➡️ {predicted_value}"
)

print(
    "\nPrediction Confidence:"
)

print(
    f"➡️ {confidence:.2f}%"
)


print("\n========================================")
print("ALL CLASS PROBABILITIES")
print("========================================")


for class_name, probability in zip(
    target_encoder.classes_,
    probabilities
):

    print(
        f"{class_name}: "
        f"{probability * 100:.2f}%"
    )


print("\n========================================")
print("TEST COMPLETED")
print("========================================")

Training Shape: (21000, 36)
Testing Shape : (4800, 36)

TRAINING MODEL...
MODEL TRAINING COMPLETED!

MODEL ACCURACY
Accuracy: 100.00%



       KIDNEY DISEASE PREDICTION

Actual Disease:
➡️ Healthy Kidney

Predicted Disease:
➡️ Healthy Kidney

Prediction Confidence:
➡️ 100.00%

ALL CLASS PROBABILITIES
Healthy Kidney: 100.00%
Kidney Failure (Stage 5): 0.00%
Mild CKD (Stage 1–2): 0.00%
Moderate CKD (Stage 3): 0.00%
Severe CKD (Stage 4): 0.00%

TEST COMPLETED


In [1]:
# ============================================================
# KIDNEY DISEASE - 5 CLASS MANUAL TESTING
# ============================================================

import pandas as pd
import numpy as np
import pickle


# ============================================================
# 1. PATHS
# ============================================================

MODEL_PATH = r"A:\Dataset\kidney_disease_model.pkl"
DATASET_PATH = r"A:\Dataset\Training_CKD_dataset.csv"


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv(DATASET_PATH)


# ============================================================
# 3. LOAD TRAINED MODEL
# ============================================================

with open(MODEL_PATH, "rb") as f:
    model_data = pickle.load(f)

model = model_data["model"]

# Different pickle versions support
if "feature_encoders" in model_data:
    encoders = model_data["feature_encoders"]
elif "encoders" in model_data:
    encoders = model_data["encoders"]
else:
    encoders = {}

target_encoder = model_data["target_encoder"]
feature_columns = model_data["feature_columns"]


# ============================================================
# 4. DISPLAY 5 CLASSES
# ============================================================

print("\n")
print("=" * 70)
print("             KIDNEY DISEASE - 5 CLASS TEST")
print("=" * 70)

print("\nAvailable Kidney Disease Classes:")

for i, cls in enumerate(target_encoder.classes_, start=1):
    print(f"{i}. {cls}")


# ============================================================
# 5. MANUAL PATIENT INPUT
# ============================================================

patient_data = {}

print("\n")
print("=" * 70)
print("                 ENTER PATIENT DETAILS")
print("=" * 70)


for feature in feature_columns:

    # --------------------------------------------------------
    # CATEGORICAL FEATURE
    # --------------------------------------------------------

    if feature in encoders:

        # Get exact values from training dataset
        available_values = (
            df[feature]
            .dropna()
            .unique()
            .tolist()
        )

        print(f"\n{feature}")
        print("-" * 40)

        for i, value in enumerate(available_values, start=1):
            print(f"{i}. {value}")

        while True:

            try:

                choice = int(
                    input("Select option: ")
                )

                if 1 <= choice <= len(available_values):

                    selected_value = available_values[
                        choice - 1
                    ]

                    patient_data[feature] = selected_value

                    break

                else:

                    print("❌ Invalid option.")

            except ValueError:

                print("❌ Enter a valid option number.")


    # --------------------------------------------------------
    # NUMERIC FEATURE
    # --------------------------------------------------------

    else:

        while True:

            try:

                value = float(
                    input(f"Enter {feature}: ")
                )

                patient_data[feature] = value

                break

            except ValueError:

                print("❌ Enter a valid numeric value.")


# ============================================================
# 6. CREATE DATAFRAME
# ============================================================

input_df = pd.DataFrame(
    [patient_data]
)

# IMPORTANT:
# Same feature order used during training

input_df = input_df[
    feature_columns
]


# ============================================================
# 7. APPLY FEATURE ENCODERS
# ============================================================

for feature, encoder in encoders.items():

    if feature in input_df.columns:

        value = input_df[feature].iloc[0]

        try:

            input_df[feature] = encoder.transform(
                [value]
            )

        except ValueError:

            print("\n❌ Invalid categorical value.")
            print(
                f"{feature} = {value}"
            )

            print(
                "Model accepted values:",
                list(encoder.classes_)
            )

            raise


# ============================================================
# 8. MAKE PREDICTION
# ============================================================

prediction = model.predict(
    input_df
)


# ============================================================
# 9. CONVERT PREDICTION TO DISEASE NAME
# ============================================================

predicted_class = (
    target_encoder.inverse_transform(
        prediction
    )[0]
)


# ============================================================
# 10. PREDICTION PROBABILITY
# ============================================================

probabilities = model.predict_proba(
    input_df
)[0]


# ============================================================
# 11. CONFIDENCE
# ============================================================

confidence = (
    np.max(probabilities) * 100
)


# ============================================================
# 12. RESULT
# ============================================================

print("\n\n")
print("=" * 70)
print("                    PREDICTION RESULT")
print("=" * 70)

print("\n🩺 Kidney Disease / Stage")
print("----------------------------------------")
print(f"➡️ {predicted_class}")

print("\n📊 Prediction Confidence")
print("----------------------------------------")
print(f"➡️ {confidence:.2f}%")


# ============================================================
# 13. ALL 5 CLASS PROBABILITIES
# ============================================================

print("\n")
print("=" * 70)
print("                  5 CLASS PROBABILITIES")
print("=" * 70)

for class_name, probability in zip(
    target_encoder.classes_,
    probabilities
):

    print(
        f"{class_name:<35} : "
        f"{probability * 100:.2f}%"
    )


# ============================================================
# 14. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)

if "Healthy" in predicted_class:

    print("✅ RESULT: Healthy Kidney")

else:

    print(
        f"⚠️ RESULT: {predicted_class}"
    )

print("=" * 70)



             KIDNEY DISEASE - 5 CLASS TEST

Available Kidney Disease Classes:
1. Healthy Kidney
2. Kidney Failure (Stage 5)
3. Mild CKD (Stage 1–2)
4. Moderate CKD (Stage 3)
5. Severe CKD (Stage 4)


                 ENTER PATIENT DETAILS


Enter Age:  45
Enter Gender:  Female


❌ Enter a valid numeric value.


Enter Gender:  2
Enter BMI:  56
Enter Systolic_BP:  89
Enter Diastolic_BP:  39
Enter Heart_Rate:  56
Enter Serum_Creatinine:  78
Enter Blood_Urea_Nitrogen:  23
Enter eGFR:  45
Enter Urine_Albumin:  67
Enter Urine_Protein:  48
Enter Albumin_Creatinine_Ratio:  48
Enter Urine_Specific_Gravity:  47
Enter Sodium:  57u


❌ Enter a valid numeric value.


...


Family_History_Kidney
----------------------------------------
1. Yes
2. No


Select option:  5


❌ Invalid option.


Select option:  1





                    PREDICTION RESULT

🩺 Kidney Disease / Stage
----------------------------------------
➡️ Mild CKD (Stage 1–2)

📊 Prediction Confidence
----------------------------------------
➡️ 41.33%


                  5 CLASS PROBABILITIES
Healthy Kidney                      : 29.00%
Kidney Failure (Stage 5)            : 6.67%
Mild CKD (Stage 1–2)                : 41.33%
Moderate CKD (Stage 3)              : 19.33%
Severe CKD (Stage 4)                : 3.67%


⚠️ RESULT: Mild CKD (Stage 1–2)


In [2]:
# ============================================================
# KIDNEY DISEASE - 5 CLASS PREDICTION
# USING ONLY 10 IMPORTANT FEATURES
# ============================================================

import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 1. DATASET PATH
# ============================================================

TRAIN_PATH = r"A:\Dataset\Training_CKD_dataset.csv"


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv(TRAIN_PATH)

print("Dataset Shape:", df.shape)


# ============================================================
# 3. SELECT ONLY 10 IMPORTANT FEATURES
# ============================================================

FEATURES = [
    "Age",
    "Gender",
    "BMI",
    "Systolic_BP",
    "Serum_Creatinine",
    "Blood_Urea_Nitrogen",
    "eGFR",
    "Urine_Albumin",
    "Albumin_Creatinine_Ratio",
    "Diabetes"
]

TARGET = "Target"


# ============================================================
# 4. CHECK FEATURES
# ============================================================

print("\nSelected Features:")

for feature in FEATURES:
    print("✔", feature)


# ============================================================
# 5. CREATE DATA
# ============================================================

X = df[FEATURES].copy()
y = df[TARGET].copy()


# ============================================================
# 6. ENCODE DIABETES
# ============================================================

diabetes_encoder = LabelEncoder()

X["Diabetes"] = diabetes_encoder.fit_transform(
    X["Diabetes"].astype(str)
)


# ============================================================
# 7. ENCODE TARGET
# ============================================================

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(
    y.astype(str)
)


# ============================================================
# 8. TRAIN MODEL
# ============================================================

print("\n")
print("=" * 60)
print("          TRAINING RANDOM FOREST MODEL")
print("=" * 60)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X, y_encoded)


# ============================================================
# 9. TRAINING ACCURACY
# ============================================================

train_prediction = model.predict(X)

accuracy = accuracy_score(
    y_encoded,
    train_prediction
)

print("\nTraining Accuracy:")
print(f"{accuracy * 100:.2f}%")


# ============================================================
# 10. SHOW 5 CLASSES
# ============================================================

print("\n")
print("=" * 60)
print("             5 KIDNEY CLASSES")
print("=" * 60)

for i, class_name in enumerate(
    target_encoder.classes_,
    start=1
):
    print(f"{i}. {class_name}")


# ============================================================
# 11. SAVE MODEL
# ============================================================

model_data = {
    "model": model,
    "diabetes_encoder": diabetes_encoder,
    "target_encoder": target_encoder,
    "feature_columns": FEATURES
}

MODEL_PATH = r"A:\Dataset\kidney_10_features_model.pkl"

with open(MODEL_PATH, "wb") as file:
    pickle.dump(model_data, file)

print("\nModel saved successfully:")
print(MODEL_PATH)


# ============================================================
# 12. MANUAL PATIENT TESTING
# ============================================================

print("\n")
print("=" * 60)
print("          MANUAL PATIENT TESTING")
print("=" * 60)


patient = {}


# ------------------------------------------------------------
# AGE
# ------------------------------------------------------------

patient["Age"] = float(
    input("\nEnter Age: ")
)


# ------------------------------------------------------------
# GENDER
# ------------------------------------------------------------

print("\nGender")
print("0 = Female")
print("1 = Male")

patient["Gender"] = float(
    input("Enter Gender (0/1): ")
)


# ------------------------------------------------------------
# BMI
# ------------------------------------------------------------

patient["BMI"] = float(
    input("\nEnter BMI: ")
)


# ------------------------------------------------------------
# SYSTOLIC BP
# ------------------------------------------------------------

patient["Systolic_BP"] = float(
    input("\nEnter Systolic BP: ")
)


# ------------------------------------------------------------
# SERUM CREATININE
# ------------------------------------------------------------

patient["Serum_Creatinine"] = float(
    input("\nEnter Serum Creatinine: ")
)


# ------------------------------------------------------------
# BLOOD UREA NITROGEN
# ------------------------------------------------------------

patient["Blood_Urea_Nitrogen"] = float(
    input("\nEnter Blood Urea Nitrogen (BUN): ")
)


# ------------------------------------------------------------
# eGFR
# ------------------------------------------------------------

patient["eGFR"] = float(
    input("\nEnter eGFR: ")
)


# ------------------------------------------------------------
# URINE ALBUMIN
# ------------------------------------------------------------

patient["Urine_Albumin"] = float(
    input("\nEnter Urine Albumin: ")
)


# ------------------------------------------------------------
# ACR
# ------------------------------------------------------------

patient["Albumin_Creatinine_Ratio"] = float(
    input("\nEnter Albumin Creatinine Ratio (ACR): ")
)


# ------------------------------------------------------------
# DIABETES
# ------------------------------------------------------------

print("\nDiabetes")
print("0 = No")
print("1 = Yes")

diabetes_input = input(
    "Enter Diabetes (0/1): "
)

if diabetes_input == "0":
    patient["Diabetes"] = "No"
else:
    patient["Diabetes"] = "Yes"


# ============================================================
# 13. CREATE INPUT DATAFRAME
# ============================================================

input_df = pd.DataFrame(
    [patient]
)

input_df = input_df[FEATURES]


# ============================================================
# 14. ENCODE DIABETES
# ============================================================

input_df["Diabetes"] = diabetes_encoder.transform(
    input_df["Diabetes"].astype(str)
)


# ============================================================
# 15. PREDICTION
# ============================================================

prediction = model.predict(
    input_df
)


# ============================================================
# 16. DISEASE NAME
# ============================================================

predicted_disease = target_encoder.inverse_transform(
    prediction
)[0]


# ============================================================
# 17. CONFIDENCE
# ============================================================

probabilities = model.predict_proba(
    input_df
)[0]

confidence = np.max(probabilities) * 100


# ============================================================
# 18. FINAL RESULT
# ============================================================

print("\n\n")
print("=" * 60)
print("              PREDICTION RESULT")
print("=" * 60)

print("\n🩺 Kidney Disease:")
print(f"➡️ {predicted_disease}")

print("\n📊 Prediction Confidence:")
print(f"➡️ {confidence:.2f}%")


# ============================================================
# 19. ALL 5 CLASS PROBABILITIES
# ============================================================

print("\n")
print("=" * 60)
print("             CLASS PROBABILITIES")
print("=" * 60)

for class_name, probability in zip(
    target_encoder.classes_,
    probabilities
):

    print(
        f"{class_name:<35} : "
        f"{probability * 100:.2f}%"
    )


print("\n")
print("=" * 60)
print("                TEST COMPLETED")
print("=" * 60)

Dataset Shape: (21000, 36)

Selected Features:
✔ Age
✔ Gender
✔ BMI
✔ Systolic_BP
✔ Serum_Creatinine
✔ Blood_Urea_Nitrogen
✔ eGFR
✔ Urine_Albumin
✔ Albumin_Creatinine_Ratio
✔ Diabetes


          TRAINING RANDOM FOREST MODEL

Training Accuracy:
100.00%


             5 KIDNEY CLASSES
1. Healthy Kidney
2. Kidney Failure (Stage 5)
3. Mild CKD (Stage 1–2)
4. Moderate CKD (Stage 3)
5. Severe CKD (Stage 4)

Model saved successfully:
A:\Dataset\kidney_10_features_model.pkl


          MANUAL PATIENT TESTING



Enter Age:  45



Gender
0 = Female
1 = Male


Enter Gender (0/1):  1

Enter BMI:  67

Enter Systolic BP:  34

Enter Serum Creatinine:  78

Enter Blood Urea Nitrogen (BUN):  45

Enter eGFR:  67

Enter Urine Albumin:  34

Enter Albumin Creatinine Ratio (ACR):  67



Diabetes
0 = No
1 = Yes


Enter Diabetes (0/1):  1





              PREDICTION RESULT

🩺 Kidney Disease:
➡️ Mild CKD (Stage 1–2)

📊 Prediction Confidence:
➡️ 51.00%


             CLASS PROBABILITIES
Healthy Kidney                      : 10.33%
Kidney Failure (Stage 5)            : 6.33%
Mild CKD (Stage 1–2)                : 51.00%
Moderate CKD (Stage 3)              : 25.67%
Severe CKD (Stage 4)                : 6.67%


                TEST COMPLETED
